In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

In [3]:
# --- Configurações ---
MODEL_PATH = "modelo_completo.keras"      # ajuste para o caminho do seu modelo salvo
CASCADE_PATH = "haarcascade_frontalface_default.xml"
IMG_SIZE = 48                              # mesmo tamanho usado no treino

In [ ]:
# Ordem das classes igual ao FER-2013 (confirme se bate com a ordem usada
# no seu flow_from_directory / ImageDataGenerator - ela é alfabética por padrão)
CLASSES = ["Raiva", "Desgosto", "Medo", "Feliz", "Neutro", "Triste", "Surpresa"]

# --- Carregar modelo e cascade ---
#print("Carregando modelo...")
#model = load_model(MODEL_PATH)

import json
import zipfile
import keras
from keras.models import load_model


# Função mágica para limpar os parâmetros incompatíveis do dicionário
def limpar_config_incompativel(d):
    if isinstance(d, dict):
        # Chaves que queremos deletar das camadas
        chaves_para_remover = ['quantization_config', 'lora_rank', 'lora_alpha']
        for chave in chaves_para_remover:
            d.pop(chave, None)
        # Continua procurando dentro de sub-dicionários
        for k, v in d.items():
            limpar_config_incompativel(v)
    elif isinstance(d, list):
        # Continua procurando dentro de listas (como a lista de camadas)
        for item in d:
            limpar_config_incompativel(item)

# 1. Abre o arquivo .keras e lê o JSON original como dicionário nativo
with zipfile.ZipFile(MODEL_PATH, 'r') as zf:
    config_bytes = zf.read('config.json')
    config_dict = json.loads(config_bytes.decode('utf-8'))

# 2. Aplica a limpeza cirúrgica no dicionário (sem quebrar a sintaxe!)
limpar_config_incompativel(config_dict)

# 3. Reconstroi a arquitetura do modelo com as configurações limpas
model = keras.saving.deserialize_keras_object(config_dict)

# 4. Carrega os pesos originais do arquivo para a nova estrutura limpa
model.load_weights(MODEL_PATH)

print("Maravilha! Modelo carregado com absoluto sucesso!")

🎉 Maravilha! Modelo carregado com absoluto sucesso!


In [14]:
face_cascade = cv2.CascadeClassifier(CASCADE_PATH)

In [15]:
if face_cascade.empty():
    raise IOError(f"Não foi possível carregar o Haar Cascade em: {CASCADE_PATH}")

# --- Iniciar webcam ---
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise IOError("Não foi possível acessar a webcam. Verifique se outra aplicação já está usando-a.")

print("Pressione 'q' para sair.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Falha ao capturar frame da webcam.")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    for (x, y, w, h) in faces:
        # Recorta o rosto detectado
        face_roi = gray[y:y + h, x:x + w]

        # Pré-processamento IDÊNTICO ao usado no treino:
        # resize -> normalização [0,1] -> reshape para (1, 48, 48, 1)
        face_resized = cv2.resize(face_roi, (IMG_SIZE, IMG_SIZE))
        face_normalized = face_resized.astype("float32") / 255.0
        face_input = np.reshape(face_normalized, (1, IMG_SIZE, IMG_SIZE, 1))

        # Predição
        predictions = model.predict(face_input, verbose=0)
        class_index = int(np.argmax(predictions))
        confidence = float(np.max(predictions))
        label = f"{CLASSES[class_index]} ({confidence*100:.1f}%)"

        # Desenha o retângulo e o texto no frame original
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(
            frame, label, (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
        )

    cv2.imshow("Reconhecimento de Emoção - pressione 'q' para sair", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Pressione 'q' para sair.
